# SIH NB8: foreign-only domain generalization revamp

This notebook never loads India. It selects among four predefined variants by nested held-out-country average precision, then fits a three-seed foreign ensemble.


In [ ]:
%pip install -q lightgbm==4.6.0 scikit-learn==1.6.1 pandas numpy scipy pyarrow pyproj openpyxl


In [ ]:
from pathlib import Path
import sys

CODE = Path('/kaggle/working/nb8_code')
CODE.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_common.py
"""SIH26162 shared config. Works on Kaggle (/kaggle/input/...) and locally."""
from pathlib import Path
import re, os
import numpy as np, pandas as pd

def _find_root():
    env = os.environ.get("SIH_DATA")
    if env: return Path(env)
    for base in [Path("/kaggle/input"), Path("."), Path("..")]:
        if not base.exists(): continue
        for p in base.rglob("firms/modis"):
            return p.parents[1]          # -> the dir containing firms/, eog/, facilities/
    raise FileNotFoundError("could not locate data/ root")

DATA = _find_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(exist_ok=True, parents=True)
CACHE = WORK / "cache"; OUT = WORK / "outputs"
CACHE.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

# Country roles are FIXED by the data explainer, Sec 1.1. India is never trained on.
TRAIN_POS = ["Iraq", "Algeria", "Nigeria", "Libya"]      # flare-dense -> EOG positives
TRAIN_BKG = ["Angola", "Indonesia"]                       # vegetation-fire background
TRAIN_COUNTRIES = TRAIN_POS + TRAIN_BKG
HOLDOUT = "India"

# NOAA-20 is absent for Angola and Indonesia -- never build raw 3-sensor ratios.
SENSORS_BY_COUNTRY = {c: {"MODIS", "VIIRS_SNPP", "VIIRS_N20"} for c in
                      ["India", "Iraq", "Algeria", "Nigeria", "Libya"]}
SENSORS_BY_COUNTRY.update({c: {"MODIS", "VIIRS_SNPP"} for c in ["Angola", "Indonesia"]})

INSTRUMENT_DIRS = {"modis": "MODIS", "viirs_snpp": "VIIRS_SNPP", "viirs_noaa20": "VIIRS_N20"}
FNAME_RE = re.compile(r"(?P<prefix>[a-z0-9-]+)_(?P<year>\d{4})_(?P<country>[A-Za-z]+)\.csv$")

# acq_time MUST stay string ("0136" -> 01:36); reading as int corrupts it.
RAW_DTYPES = {
    "latitude": "float64", "longitude": "float64",
    "scan": "float32", "track": "float32", "frp": "float32",
    "brightness": "float32", "bright_t31": "float32",
    "bright_ti4": "float32", "bright_ti5": "float32",
    "acq_time": "string", "satellite": "string", "instrument": "string",
    "confidence": "string", "version": "string", "daynight": "string", "type": "float32",
}

def firms_files(instrument=None):
    dirs = [instrument] if instrument else list(INSTRUMENT_DIRS)
    out = []
    for d in dirs:
        # NB: NOAA-20 files are named viirs-jpss1_*, NOT viirs-noaa20_*
        for p in sorted((DATA / "firms" / d).glob("*.csv")):
            m = FNAME_RE.search(p.name)
            if not m: raise ValueError(f"unparseable FIRMS filename: {p}")
            out.append(dict(path=p, instdir=d, sensor=INSTRUMENT_DIRS[d],
                            year=int(m.group("year")), country=m.group("country")))
    return out

def read_firms_csv(path):
    """Read one FIRMS CSV, unifying the MODIS and VIIRS schemas."""
    df = pd.read_csv(path, dtype=RAW_DTYPES)
    if "bright_t31" in df.columns:                      # MODIS: 4um / 11um
        df = df.rename(columns={"brightness": "t_mir", "bright_t31": "t_lwir"})
    else:                                               # VIIRS: I4 / I5
        df = df.rename(columns={"bright_ti4": "t_mir", "bright_ti5": "t_lwir"})
    return df

def load_firms(countries=None, sensors=None, columns=None):
    """Load FIRMS into one tidy frame. ~19.3M rows total; fine in Kaggle's 30 GB."""
    fs = firms_files()
    if countries: fs = [f for f in fs if f["country"] in countries]
    if sensors:   fs = [f for f in fs if f["sensor"] in sensors]
    out = []
    for f in fs:
        d = read_firms_csv(f["path"])
        d["acq_dt"] = pd.to_datetime(d.acq_date, format="%Y-%m-%d")
        d["acq_time"] = d.acq_time.str.zfill(4).astype("int16")
        d["sensor"] = f["sensor"]; d["country"] = f["country"]
        d["year"] = np.int16(f["year"]); d["type"] = d["type"].astype("int8")
        d = d.drop(columns=["instrument", "acq_date"])
        if columns: d = d[columns]
        out.append(d)
    df = pd.concat(out, ignore_index=True)
    for c in ["sensor", "country", "satellite", "confidence", "daynight"]:
        if c in df: df[c] = df[c].astype("category")
    return df

def eog_sites(active_years=range(2019, 2025)):
    """Verified positive class: EOG/World Bank flare sites active in the FIRMS window."""
    p = DATA / "eog" / "flare_inventory" / \
        "Flare-Volume-Estimates-by-individual-Flare-Location-2012-2025.xlsx"
    w = pd.read_excel(p)
    yrs = [y for y in active_years if y in w.columns]
    w["vol"] = w[yrs].sum(axis=1, min_count=1)
    w["active"] = (w[yrs].fillna(0) > 0).any(axis=1)
    w = w.rename(columns={"Flare id": "flare_id", "Latitude": "lat", "Longitude": "lon",
                          "Country": "country", "Location": "location",
                          "Field Type": "field_type"})
    return w.loc[w.active, ["flare_id", "country", "lat", "lon", "location",
                            "field_type", "vol"]].reset_index(drop=True)


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_eval.py
"""Shared evaluation: one metric block for every experiment, plus split builders."""
import time, json
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, precision_score, recall_score, average_precision_score,
                             roc_auc_score, matthews_corrcoef, confusion_matrix,
                             brier_score_loss, precision_recall_curve)
from sklearn.model_selection import GroupKFold
from kg_common import *

def metrics(y, p, thr=0.5, train_s=None, infer_s=None, name="", extra=None):
    yh = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    d = dict(
        experiment=name, n=len(y), n_pos=int(y.sum()), thr=round(float(thr), 4),
        precision=precision_score(y, yh, zero_division=0),
        recall=recall_score(y, yh, zero_division=0),
        f1=f1_score(y, yh, zero_division=0),
        pr_auc=average_precision_score(y, p) if y.sum() else np.nan,
        roc_auc=roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan,
        mcc=matthews_corrcoef(y, yh) if len(set(yh)) > 1 else 0.0,
        brier=brier_score_loss(y, np.clip(p, 0, 1)),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
        train_s=round(train_s, 2) if train_s is not None else np.nan,
        infer_s=round(infer_s, 3) if infer_s is not None else np.nan)
    if extra: d.update(extra)
    return d

def best_f1_threshold(y, p, grid=None):
    """Return the exact score threshold that maximizes binary F1.

    The previous default searched 120 score quantiles. With fewer than 0.3%
    labelled positives, that grid skipped most of the operational score tail
    and could miss materially better thresholds. An explicit ``grid`` is still
    supported for sensitivity analysis; the default now evaluates every
    precision-recall operating point.
    """
    y = np.asarray(y, dtype="int8")
    p = np.asarray(p, dtype="float64")
    if len(y) != len(p) or not len(y):
        raise ValueError("y and p must be non-empty arrays of equal length")

    if grid is not None:
        grid = np.asarray(grid, dtype="float64")
        if not len(grid):
            raise ValueError("grid cannot be empty")
        fs = np.asarray([
            f1_score(y, p >= t, zero_division=0) for t in grid
        ])
        j = int(np.nanargmax(fs))
        return float(grid[j]), float(fs[j])

    if y.sum() == 0:
        return float(np.nextafter(p.max(), np.inf)), 0.0

    precision, recall, thresholds = precision_recall_curve(y, p)
    if not len(thresholds):
        return 0.5, 0.0
    f1 = (2 * precision[:-1] * recall[:-1] /
          np.maximum(precision[:-1] + recall[:-1], 1e-15))
    j = int(np.nanargmax(f1))
    return float(thresholds[j]), float(f1[j])

def calibration_table(y, p, bins=10):
    q = pd.qcut(pd.Series(p), bins, duplicates="drop", labels=False)
    t = pd.DataFrame({"y": y, "p": p, "b": q}).groupby("b").agg(
        n=("y", "size"), pred=("p", "mean"), obs=("y", "mean"))
    t["gap"] = (t.pred - t.obs).abs()
    return t

def load_features(countries, tag=None):
    suffix = f"_{tag}" if tag else ""
    fs = [pd.read_parquet(CACHE / f"features_{c}{suffix}.parquet") for c in countries]
    return pd.concat(fs, ignore_index=True)

def grouped_folds(df, n_splits=5, seed=0):
    """GroupKFold on block_id -- fragments of one physical flare stay on one side."""
    gk = GroupKFold(n_splits=n_splits)
    return list(gk.split(df, df.is_eog_flare.values, groups=df.block_id.values))

def leave_country_out(df, countries=None):
    cs = countries or sorted(df.country.unique())
    for c in cs:
        te = (df.country == c).values
        if df.loc[te, "is_eog_flare"].sum() == 0:
            continue                     # no positives -> nothing to score
        yield c, ~te, te

def log_experiment(rows, path):
    p = OUT / path
    df = pd.DataFrame(rows)
    df.to_csv(p, index=False)
    return df

def show(df, cols=None):
    cols = cols or ["experiment", "n", "n_pos", "precision", "recall", "f1",
                    "pr_auc", "roc_auc", "mcc", "brier", "tp", "fp", "fn", "train_s"]
    d = df[[c for c in cols if c in df.columns]].copy()
    for c in d.columns:
        if d[c].dtype.kind == "f": d[c] = d[c].round(4)
    return d.to_string(index=False)


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_02_source_clustering.py
"""Stage 02: collapse FIRMS detections into persistent thermal SOURCES, and attach
EOG labels at source level.

Unit of analysis = one physical thermal source, not one detection row.

Primary method is metric grid snapping (deterministic, order-independent, and it
CANNOT chain -- important, because 8-neighbour agglomeration merges a flare into an
adjacent wildfire front during India's burning season). DBSCAN is run as a
sensitivity check only.

Writes: cache/sources_<country>.parquet, cache/detections_<country>.parquet
"""
import sys, time, json
import numpy as np, pandas as pd
from kg_common import *

GRID_M      = 1000       # source cell size in metres (chosen by sensitivity sweep)
LABEL_R_M   = 1000       # EOG match radius, metres
BLOCK_M     = 10_000     # spatial block used as the CV grouping unit (anti-leakage)
R_EARTH     = 6_371_000.0

def to_local_m(lat, lon, lat0, lon0):
    """Equirectangular projection to metres about (lat0, lon0). Accurate to <0.1%
    over a single country, which is far below the 500 m grid."""
    x = np.radians(lon - lon0) * R_EARTH * np.cos(np.radians(lat0))
    y = np.radians(lat - lat0) * R_EARTH
    return x, y

def grid_ids(x, y, cell):
    ix = np.floor(x / cell).astype("int64")
    iy = np.floor(y / cell).astype("int64")
    return ix, iy, (ix + 2_000_000) * 4_000_000 + (iy + 2_000_000)

def attach_eog_labels(src, country, lat0, lon0, grid_m=GRID_M,
                      label_r=LABEL_R_M, active_years=range(2019, 2025)):
    """Attach nearest-site EOG labels to a source table.

    ``src`` must contain ``x`` and ``y`` coordinates in the local projection
    defined by ``lat0`` and ``lon0``. Restricting ``active_years`` is essential
    for common-window experiments: a flare active outside the observation
    window must not become a positive label inside it.
    """
    src = src.copy()
    e = eog_sites(active_years=active_years)
    e = e[e.country == country].reset_index(drop=True)
    if not len(e):
        src["eog_dist_m"] = np.inf
        src["is_eog_flare"] = np.int8(0)
        src["eog_flare_id"] = None
        src["eog_offshore"] = False
        return src

    ex, ey = to_local_m(e.lat.values, e.lon.values, lat0, lon0)
    eix, eiy, _ = grid_ids(ex, ey, grid_m)
    buckets = {}
    for j, (a, b) in enumerate(zip(eix, eiy)):
        buckets.setdefault((a, b), []).append(j)

    six = np.floor(src.x.values / grid_m).astype("int64")
    siy = np.floor(src.y.values / grid_m).astype("int64")
    best_d = np.full(len(src), np.inf)
    best_j = np.full(len(src), -1)
    rad = int(np.ceil(label_r / grid_m))
    for dx in range(-rad, rad + 1):
        for dy in range(-rad, rad + 1):
            cand = {}
            for i, (a, b) in enumerate(zip(six + dx, siy + dy)):
                js = buckets.get((a, b))
                if js:
                    cand[i] = js
            for i, js in cand.items():
                d = np.hypot(ex[js] - src.x.values[i], ey[js] - src.y.values[i])
                k = int(np.argmin(d))
                if d[k] < best_d[i]:
                    best_d[i] = d[k]
                    best_j[i] = js[k]

    src["eog_dist_m"] = best_d
    src["is_eog_flare"] = (best_d <= label_r).astype("int8")
    safe_j = np.clip(best_j, 0, None)
    src["eog_flare_id"] = np.where(
        best_j >= 0, e.flare_id.values[safe_j], None)
    src["eog_offshore"] = np.where(
        best_j >= 0, e.location.values[safe_j] == "OFFSHORE", False)
    src.loc[src.is_eog_flare == 0, "eog_flare_id"] = None
    return src

def build_country(country, grid_m=GRID_M, label_r=LABEL_R_M, verbose=True):
    t0 = time.time()
    df = load_firms(countries=[country])
    lat0, lon0 = df.latitude.mean(), df.longitude.mean()
    x, y = to_local_m(df.latitude.values, df.longitude.values, lat0, lon0)
    ix, iy, key = grid_ids(x, y, grid_m)
    codes, uniq = pd.factorize(key)
    df["source_idx"] = codes
    df["_x"], df["_y"] = x, y

    n_src = len(uniq)
    if verbose:
        print(f"{country}: {len(df):,} detections -> {n_src:,} sources "
              f"({len(df)/n_src:.1f} det/source)  [{time.time()-t0:.0f}s]", flush=True)

    # ---- source-level geometry (features come later, in stage 03) ----
    g = df.groupby("source_idx", observed=True)
    src = pd.DataFrame({
        "n_det":   g.size(),
        "lat":     g.latitude.mean(),
        "lon":     g.longitude.mean(),
        "x":       g._x.mean(),
        "y":       g._y.mean(),
        "date_min": g.acq_dt.min(),
        "date_max": g.acq_dt.max(),
        "n_days":  g.acq_dt.apply(lambda s: s.dt.normalize().nunique()),
    })
    src["span_days"] = (src.date_max - src.date_min).dt.days
    src["country"] = country
    # Grid snapping fragments ONE physical flare into ~4 sources (measured: 3.9-4.2x).
    # Fragments of the same site must never straddle a train/val boundary, so all
    # splitting is done on these coarse blocks, never on source_id.
    bix = np.floor(src.x.values / BLOCK_M).astype("int64")
    biy = np.floor(src.y.values / BLOCK_M).astype("int64")
    src["block_id"] = country + "_b" + pd.Series((bix + 100000) * 1000000 + (biy + 100000)).astype(str)
    src["source_id"] = country + "_" + src.index.astype(str)
    src = src.reset_index(drop=True)

    # ---- EOG labels: match each source to the nearest active flare site ----
    src = attach_eog_labels(src, country, lat0, lon0, grid_m, label_r)

    det = df[["source_idx", "latitude", "longitude", "t_mir", "t_lwir", "frp", "scan",
              "track", "acq_dt", "acq_time", "satellite", "confidence", "daynight",
              "type", "sensor", "year"]].copy()
    det["source_id"] = src.source_id.values[det.source_idx.values]
    det = det.drop(columns=["source_idx"])

    src.to_parquet(CACHE / f"sources_{country}.parquet", index=False)
    det.to_parquet(CACHE / f"detections_{country}.parquet", index=False)
    return src

def main(countries=None):
    countries = countries or (TRAIN_COUNTRIES + [HOLDOUT])
    rows = []
    for c in countries:
        s = build_country(c)
        matched = int(s.is_eog_flare.sum())
        e_n = int((eog_sites().country == c).sum())
        rows.append(dict(
            country=c, detections=int(s.n_det.sum()), sources=len(s),
            det_per_source=round(s.n_det.sum() / len(s), 1),
            src_1det=int((s.n_det == 1).sum()),
            src_1det_pct=round((s.n_det == 1).mean() * 100, 1),
            src_ge5days=int((s.n_days >= 5).sum()),
            src_ge30days=int((s.n_days >= 30).sum()),
            src_multiyear=int((s.span_days >= 365).sum()),
            eog_sites=e_n, sources_matched=matched,
            eog_recovered=int(s.loc[s.is_eog_flare == 1, "eog_flare_id"].nunique()),
            eog_recall_pct=round(s.loc[s.is_eog_flare == 1, "eog_flare_id"].nunique()
                                 / max(e_n, 1) * 100, 1),
            pos_rate_pct=round(matched / len(s) * 100, 3)))
        print(pd.DataFrame(rows[-1:]).to_string(index=False), flush=True)
    rep = pd.DataFrame(rows)
    rep.to_csv(OUT / "02_source_summary.csv", index=False)
    print("\n" + "=" * 120); print("SOURCE-LEVEL SUMMARY"); print("=" * 120)
    print(rep.to_string(index=False))
    print("\nNOTE: sources with is_eog_flare=0 are UNLABELLED, not negative. EOG covers"
          "\ngas flares only (>~1100 C); kilns, cement and steel are industrial but absent.")
    return rep

if __name__ == "__main__":
    main(sys.argv[1:] or None)


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_03_features.py
"""Stage 03: source-level features from FIRMS only.

Leakage rules enforced here:
  * NO raw lat/lon         -- near-perfect country proxy, and country correlates with label
  * NO country column
  * NO NASA `type`         -- type=2 IS a persistence mask; using it reproduces the mask
  * NO eog_dist_m          -- that IS the label
  * NO NOAA-20             -- absent for Angola and Indonesia, so any count that includes it
                              encodes country identity. Model uses MODIS + VIIRS S-NPP only,
                              which every country has. N20 is kept for India-side evidence.

Writes cache/features_<country>.parquet
"""
import sys, time
import numpy as np, pandas as pd
from kg_common import *
from kg_02_source_clustering import attach_eog_labels, to_local_m

MODEL_SENSORS = ["MODIS", "VIIRS_SNPP"]      # uniform across all 7 countries
FEATURE_BLOCKLIST = {"lat", "lon", "x", "y", "country", "type", "eog_dist_m",
                     "is_eog_flare", "eog_flare_id", "eog_offshore", "block_id",
                     "source_id", "date_min", "date_max"}

def _circ(hours):
    """circular mean/std of an hour-of-day array, in hours"""
    a = hours.to_numpy() * (2 * np.pi / 24.0)
    s, c = np.sin(a).mean(), np.cos(a).mean()
    R = np.hypot(s, c)
    mean = (np.arctan2(s, c) % (2 * np.pi)) * 24 / (2 * np.pi)
    std = np.sqrt(max(-2 * np.log(max(R, 1e-12)), 0)) * 24 / (2 * np.pi)
    return mean, std

def build(country, sensors=MODEL_SENSORS, years=None, output_tag=None):
    """Build source features, optionally inside a fixed observation window.

    When ``years`` is provided, detections and EOG activity are restricted to
    exactly those years. Window-normalized persistence features are added and
    the output is written as ``features_<country>_<output_tag>.parquet``.
    """
    t0 = time.time()
    det = pd.read_parquet(CACHE / f"detections_{country}.parquet")
    det = det[det.sensor.isin(sensors)].copy()
    if years is not None:
        years = tuple(sorted(set(int(y) for y in years)))
        if not years:
            raise ValueError("years cannot be empty")
        det = det[det.year.isin(years)].copy()
    if det.empty:
        raise ValueError(
            f"no detections for {country} with sensors={sensors}, years={years}")

    det["day"]      = det.acq_dt.values.astype("datetime64[D]")
    det["doy"]      = det.acq_dt.dt.dayofyear
    det["month"]    = det.acq_dt.dt.month
    det["pix_km2"]  = det.scan * det.track
    det["frp_dens"] = det.frp / det.pix_km2
    det["dt_mir_lwir"] = det.t_mir - det.t_lwir          # separates flares from veg fires
    det["sat"]      = (det.t_mir >= 367.0).astype("int8")
    det["is_night"] = (det.daynight.astype(str) == "N").astype("int8")
    # local solar time = UTC hour + lon/15
    det["lst"] = ((det.acq_time // 100 + (det.acq_time % 100) / 60.0)
                  + det.longitude / 15.0) % 24.0

    g = det.groupby("source_id", observed=True)
    f = pd.DataFrame(index=g.size().index)
    f["n_det"] = g.size()

    # ---- intensity ----
    for col in ["frp", "frp_dens", "t_mir", "t_lwir", "dt_mir_lwir"]:
        a = g[col]
        f[f"{col}_mean"] = a.mean(); f[f"{col}_max"] = a.max()
        f[f"{col}_std"]  = a.std().fillna(0); f[f"{col}_med"] = a.median()
    f["frp_p90"] = g.frp.quantile(0.90)
    f["frp_cv"]  = f.frp_std / f.frp_mean.replace(0, np.nan)
    f["frp_sum"] = g.frp.sum()
    f["pix_km2_mean"] = g.pix_km2.mean()

    # ---- saturation / day-night ----
    f["sat_frac"]   = g.sat.mean()
    f["night_frac"] = g.is_night.mean()

    # ---- temporal persistence ----
    f["n_days"]   = g.day.nunique()
    f["n_months"] = g.acq_dt.apply(lambda s: s.dt.to_period("M").nunique())
    f["n_years"]  = g.year.nunique()
    dmin, dmax = g.day.min(), g.day.max()
    f["span_days"]  = (dmax - dmin).dt.days.astype("float32")
    f["duty_cycle"] = f.n_days / f.span_days.replace(0, np.nan)
    f["det_per_day"] = f.n_det / f.n_days

    # max gap between consecutive active days
    ud = det[["source_id", "day"]].drop_duplicates().sort_values(["source_id", "day"])
    gap = ud.groupby("source_id", observed=True).day.diff().dt.days
    f["max_gap_days"]  = gap.groupby(ud.source_id.values).max().reindex(f.index).fillna(0)
    f["mean_gap_days"] = gap.groupby(ud.source_id.values).mean().reindex(f.index).fillna(0)

    # ---- seasonality: entropy over months (flat = year-round = industrial-like) ----
    mh = (det.groupby(["source_id", "month"], observed=True).size()
             .unstack(fill_value=0).reindex(f.index, fill_value=0))
    p = mh.div(mh.sum(1).replace(0, np.nan), axis=0).fillna(0)
    f["month_entropy"] = -(p * np.log(p.where(p > 0, 1))).sum(1) / np.log(12)
    f["month_max_share"] = p.max(1)

    # ---- overpass timing ----
    lst = g.lst.apply(lambda s: pd.Series(_circ(s), index=["m", "s"]))
    f["lst_mean"] = lst.xs("m", level=1); f["lst_std"] = lst.xs("s", level=1)

    # ---- cross-instrument (MODIS vs S-NPP only; both exist for all countries) ----
    sc = (det.groupby(["source_id", "sensor"], observed=True).size()
             .unstack(fill_value=0).reindex(f.index, fill_value=0))
    for s in sensors:
        if s not in sc: sc[s] = 0
    f["n_modis"] = sc["MODIS"]; f["n_snpp"] = sc["VIIRS_SNPP"]
    # small source -> resolved by 375 m VIIRS, diluted below MODIS threshold -> high ratio
    f["snpp_modis_ratio"] = (f.n_snpp + 1) / (f.n_modis + 1)
    f["n_sensors"] = (sc[sensors] > 0).sum(1)

    # ---- confidence (two different vocabularies) ----
    cf = det.confidence.astype(str)
    mm = det.sensor.astype(str) == "MODIS"
    f["conf_modis_mean"] = (pd.to_numeric(cf.where(mm), errors="coerce")
                            .groupby(det.source_id.values).mean().reindex(f.index))
    for lv in ["n", "l", "h"]:
        f[f"conf_viirs_{lv}_frac"] = ((cf == lv).astype("int8")
                                      .groupby(det.source_id.values).mean().reindex(f.index))

    # ---- spatial spread within the source, in metres ----
    latm = g.latitude.std().fillna(0) * 111_320.0
    lonm = g.longitude.std().fillna(0) * 111_320.0 * np.cos(np.radians(g.latitude.mean()))
    f["spread_m"] = np.hypot(latm, lonm)
    f["n_pixels"] = det.groupby("source_id", observed=True).apply(
        lambda d: len(d[["latitude", "longitude"]].drop_duplicates()), include_groups=False)

    if years is not None:
        # Exposure-normalized alternatives to raw counts. The robust model
        # blocklists the raw versions and consumes these columns instead.
        n_window_years = float(len(years))
        window_start = pd.Timestamp(min(years), 1, 1)
        window_end = pd.Timestamp(max(years), 12, 31)
        window_days = float((window_end - window_start).days + 1)
        f["det_per_year"] = f.n_det / n_window_years
        f["active_days_per_year"] = f.n_days / n_window_years
        f["active_months_per_year"] = f.n_months / n_window_years
        f["frp_sum_per_year"] = f.frp_sum / n_window_years
        f["modis_per_year"] = f.n_modis / n_window_years
        f["snpp_per_year"] = f.n_snpp / n_window_years
        f["span_window_frac"] = f.span_days / max(window_days - 1.0, 1.0)

    f = f.replace([np.inf, -np.inf], np.nan).astype("float32").reset_index()

    src = pd.read_parquet(CACHE / f"sources_{country}.parquet")
    if years is None:
        out = f.merge(src[["source_id", "block_id", "country", "lat", "lon",
                           "is_eog_flare", "eog_flare_id", "eog_dist_m", "eog_offshore"]],
                      on="source_id", how="left")
    else:
        # Recompute source centroids and EOG labels inside the common window.
        # Reuse the original 10 km block_id so the spatial grouping remains
        # anchored to the Stage 02 grid.
        geo = (det.groupby("source_id", observed=True)
                 .agg(lat=("latitude", "mean"), lon=("longitude", "mean"))
                 .reset_index())
        geo = geo.merge(src[["source_id", "block_id"]], on="source_id", how="left")
        geo["country"] = country
        lat0, lon0 = float(det.latitude.mean()), float(det.longitude.mean())
        geo["x"], geo["y"] = to_local_m(
            geo.lat.values, geo.lon.values, lat0, lon0)
        geo = attach_eog_labels(
            geo, country, lat0, lon0, active_years=years)
        out = f.merge(
            geo[["source_id", "block_id", "country", "lat", "lon",
                 "is_eog_flare", "eog_flare_id", "eog_dist_m", "eog_offshore"]],
            on="source_id", how="left")

    suffix = f"_{output_tag}" if output_tag else ""
    out.to_parquet(CACHE / f"features_{country}{suffix}.parquet", index=False)
    print(f"{country}: {len(out):,} sources x {len(feature_cols(out))} features "
          f"({int(out.is_eog_flare.sum()):,} pos), years={years or 'all'} "
          f"[{time.time()-t0:.0f}s]", flush=True)
    return out

def feature_cols(df):
    return [c for c in df.columns if c not in FEATURE_BLOCKLIST]

def main(countries=None):
    for c in (countries or TRAIN_COUNTRIES + [HOLDOUT]):
        build(c)

if __name__ == "__main__":
    main(sys.argv[1:] or None)


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_05b_robust_tabular.py
"""Stage 05b: robust common-window tabular evaluation.

This stage repairs the two main issues found after the full Stage 05 run:

1. Countries had different observation horizons. Features are rebuilt on the
   shared 2022-2024 window and EOG activity is restricted to the same years.
2. LOCO thresholds were selected on held-out labels. Thresholds are now learned
   from grouped OOF predictions on non-holdout countries only.

The stage compares a reduced binary LightGBM model with a bagged
positive-unlabelled LightGBM baseline. India is never loaded by ``run()``.

Writes:
  cache/features_<country>_2022_2024.parquet
  cache/05b_oof_predictions.parquet
  cache/05b_corrected_loco_predictions.parquet
  outputs/05b_robust_models.csv
  outputs/05b_corrected_loco.csv
  outputs/05b_feature_importance.csv
  outputs/05b_feature_manifest.json
"""
import gc
import json
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd

from kg_common import *
from kg_eval import *
from kg_03_features import build as build_features, feature_cols


WINDOW_YEARS = (2022, 2023, 2024)
FEATURE_TAG = "2022_2024"

PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=40,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    verbose=-1,
    num_threads=-1,
    seed=17,
    feature_fraction_seed=17,
    bagging_seed=17,
    data_random_seed=17,
    force_col_wise=True,
)
N_ROUNDS = 500

# These columns either encode the old observation horizon directly or were
# harmful in the Stage 05 ablation. Their normalized replacements remain.
RAW_EXPOSURE_FEATURES = {
    "n_det", "n_days", "n_months", "n_years", "span_days", "frp_sum",
    "n_modis", "n_snpp", "n_pixels",
}
HARMFUL_FEATURES = {"month_entropy", "month_max_share"}


def build_common_features(countries=None):
    """Build common-window features for foreign countries only by default."""
    countries = countries or TRAIN_COUNTRIES
    rows = []
    for country in countries:
        d = build_features(
            country,
            years=WINDOW_YEARS,
            output_tag=FEATURE_TAG,
        )
        rows.append(dict(
            country=country,
            n=len(d),
            n_pos=int(d.is_eog_flare.sum()),
            eog_sites_recovered=int(
                d.loc[d.is_eog_flare == 1, "eog_flare_id"].nunique()
            ),
        ))
        del d
        gc.collect()
    report = pd.DataFrame(rows)
    report.to_csv(OUT / "05b_common_window_summary.csv", index=False)
    print("\nCOMMON-WINDOW FEATURE SUMMARY")
    print(report.to_string(index=False))
    return report


def robust_feature_cols(df):
    cols = [
        c for c in feature_cols(df)
        if c not in RAW_EXPOSURE_FEATURES | HARMFUL_FEATURES
    ]
    required = {
        "det_per_year", "active_days_per_year", "active_months_per_year",
        "frp_sum_per_year", "modis_per_year", "snpp_per_year",
        "span_window_frac",
    }
    missing = required - set(cols)
    if missing:
        raise ValueError(
            "common-window normalized features are missing: " +
            ", ".join(sorted(missing))
        )
    return sorted(cols)


def _train_binary(X, y, train_idx, params=None, rounds=N_ROUNDS):
    dtrain = lgb.Dataset(
        X[train_idx], label=y[train_idx], free_raw_data=True
    )
    return lgb.train(params or PARAMS, dtrain, num_boost_round=rounds)


def _predict_fold(X, y, train_idx, test_idx, mode="binary", pu_bags=3,
                  unlabeled_per_positive=10, seed=17, rounds=N_ROUNDS):
    """Fit one binary model or a bagged PU ensemble and predict one fold."""
    train_s = infer_s = 0.0
    if mode == "binary":
        t0 = time.time()
        model = _train_binary(X, y, train_idx, rounds=rounds)
        train_s += time.time() - t0
        t0 = time.time()
        pred = model.predict(X[test_idx])
        infer_s += time.time() - t0
        return pred, train_s, infer_s

    if mode != "pu_bagging":
        raise ValueError(f"unknown mode: {mode}")

    pos_idx = train_idx[y[train_idx] == 1]
    unl_idx = train_idx[y[train_idx] == 0]
    if not len(pos_idx):
        raise ValueError("PU fold has no labelled positives")
    n_unl = min(len(unl_idx), unlabeled_per_positive * len(pos_idx))
    pred = np.zeros(len(test_idx), dtype="float64")
    for bag in range(pu_bags):
        rng = np.random.default_rng(seed + bag)
        sampled_unl = rng.choice(unl_idx, size=n_unl, replace=False)
        sampled_train = np.concatenate([pos_idx, sampled_unl])
        rng.shuffle(sampled_train)
        bag_params = dict(PARAMS)
        bag_params.update(
            seed=seed + bag,
            feature_fraction_seed=seed + bag,
            bagging_seed=seed + bag,
            data_random_seed=seed + bag,
        )
        t0 = time.time()
        model = _train_binary(
            X, y, sampled_train, params=bag_params, rounds=rounds
        )
        train_s += time.time() - t0
        t0 = time.time()
        pred += model.predict(X[test_idx]) / pu_bags
        infer_s += time.time() - t0
    return pred, train_s, infer_s


def cv_predict(df, cols, mode="binary", n_splits=5, pu_bags=3,
               unlabeled_per_positive=10, seed=17, rounds=N_ROUNDS):
    X = df[cols].to_numpy(dtype="float32")
    y = df.is_eog_flare.to_numpy(dtype="int8")
    oof = np.zeros(len(df), dtype="float64")
    train_s = infer_s = 0.0
    for fold, (train_idx, test_idx) in enumerate(
            grouped_folds(df, n_splits=n_splits)):
        pred, ts, ins = _predict_fold(
            X, y, train_idx, test_idx, mode=mode, pu_bags=pu_bags,
            unlabeled_per_positive=unlabeled_per_positive,
            seed=seed + fold * 100,
            rounds=rounds,
        )
        oof[test_idx] = pred
        train_s += ts
        infer_s += ins
    return oof, train_s, infer_s


def fit_predict(train_df, test_df, cols, mode="binary", pu_bags=3,
                unlabeled_per_positive=10, seed=17, rounds=N_ROUNDS):
    joined = pd.concat([train_df, test_df], ignore_index=True)
    X = joined[cols].to_numpy(dtype="float32")
    y = joined.is_eog_flare.to_numpy(dtype="int8")
    train_idx = np.arange(len(train_df), dtype="int64")
    test_idx = np.arange(len(train_df), len(joined), dtype="int64")
    pred, train_s, infer_s = _predict_fold(
        X, y, train_idx, test_idx, mode=mode, pu_bags=pu_bags,
        unlabeled_per_positive=unlabeled_per_positive, seed=seed,
        rounds=rounds,
    )
    return pred, train_s, infer_s


def corrected_loco(df, cols, inner_splits=3, rounds=N_ROUNDS):
    """Strict nested LOCO with a threshold learned without the holdout."""
    rows = []
    predictions = []
    for country in sorted(df.country.unique()):
        train_df = df[df.country != country].reset_index(drop=True)
        test_df = df[df.country == country].reset_index(drop=True)

        inner_oof, inner_train_s, _ = cv_predict(
            train_df, cols, mode="binary", n_splits=inner_splits,
            seed=1000 + len(rows) * 100,
            rounds=rounds,
        )
        inner_y = train_df.is_eog_flare.to_numpy(dtype="int8")
        threshold, inner_f1 = best_f1_threshold(inner_y, inner_oof)

        pred, final_train_s, infer_s = fit_predict(
            train_df, test_df, cols, mode="binary",
            seed=2000 + len(rows) * 100,
            rounds=rounds,
        )
        test_y = test_df.is_eog_flare.to_numpy(dtype="int8")
        rows.append(metrics(
            test_y,
            pred,
            threshold,
            train_s=inner_train_s + final_train_s,
            infer_s=infer_s,
            name=f"corrected LOCO holdout={country}",
            extra=dict(
                threshold_source=(
                    f"{inner_splits}-fold grouped OOF on non-holdout countries"
                ),
                inner_oof_f1=inner_f1,
                inner_n=len(train_df),
                inner_n_pos=int(inner_y.sum()),
            ),
        ))
        predictions.append(pd.DataFrame({
            "source_id": test_df.source_id,
            "country": country,
            "is_eog_flare": test_y,
            "score": pred,
            "threshold": threshold,
            "predicted_positive": pred >= threshold,
        }))
        print(
            f"LOCO {country}: threshold={threshold:.4f}, "
            f"PR-AUC={rows[-1]['pr_auc']:.4f}, F1={rows[-1]['f1']:.4f}",
            flush=True,
        )
        del train_df, test_df, inner_oof, pred
        gc.collect()

    result = pd.DataFrame(rows)
    result.to_csv(OUT / "05b_corrected_loco.csv", index=False)
    pd.concat(predictions, ignore_index=True).to_parquet(
        CACHE / "05b_corrected_loco_predictions.parquet", index=False
    )
    return result


def run(n_splits=5, inner_splits=3, pu_bags=3,
        unlabeled_per_positive=10, rounds=N_ROUNDS):
    """Evaluate foreign countries only. India is deliberately not loaded."""
    df = load_features(TRAIN_COUNTRIES, tag=FEATURE_TAG)
    if HOLDOUT in set(df.country):
        raise AssertionError("India entered Stage 05b training data")
    cols = robust_feature_cols(df)
    y = df.is_eog_flare.to_numpy(dtype="int8")

    rows = []
    oof_frame = df[[
        "source_id", "country", "block_id", "lat", "lon", "is_eog_flare"
    ]].copy()
    thresholds = {}

    for mode, label in [
        ("binary", "reduced LightGBM"),
        ("pu_bagging", "bagged PU LightGBM"),
    ]:
        pred, train_s, infer_s = cv_predict(
            df,
            cols,
            mode=mode,
            n_splits=n_splits,
            pu_bags=pu_bags,
            unlabeled_per_positive=unlabeled_per_positive,
            rounds=rounds,
        )
        threshold, _ = best_f1_threshold(y, pred)
        thresholds[mode] = threshold
        rows.append(metrics(
            y,
            pred,
            threshold,
            train_s=train_s,
            infer_s=infer_s,
            name=label,
            extra=dict(
                n_features=len(cols),
                window="2022-2024",
                threshold_source=(
                    f"{n_splits}-fold grouped OOF on foreign countries"
                ),
                pu_bags=pu_bags if mode == "pu_bagging" else 0,
                unlabeled_per_positive=(
                    unlabeled_per_positive if mode == "pu_bagging" else 0
                ),
            ),
        ))
        oof_frame[f"score_{mode}"] = pred
        print(f"\n{label}\n{show(pd.DataFrame(rows[-1:]))}", flush=True)

    model_results = pd.DataFrame(rows)
    model_results.to_csv(OUT / "05b_robust_models.csv", index=False)
    oof_frame.to_parquet(CACHE / "05b_oof_predictions.parquet", index=False)

    loco = corrected_loco(
        df, cols, inner_splits=inner_splits, rounds=rounds
    )

    X = df[cols].to_numpy(dtype="float32")
    model = _train_binary(
        X, y, np.arange(len(df), dtype="int64"), rounds=rounds
    )
    importance = (pd.DataFrame({
        "feature": cols,
        "gain": model.feature_importance("gain"),
    }).sort_values("gain", ascending=False))
    importance.to_csv(OUT / "05b_feature_importance.csv", index=False)

    manifest = dict(
        stage="05b_robust_tabular",
        observation_years=list(WINDOW_YEARS),
        training_countries=TRAIN_COUNTRIES,
        holdout_country=HOLDOUT,
        holdout_loaded=False,
        model_sensors=["MODIS", "VIIRS_SNPP"],
        features=cols,
        excluded_raw_exposure_features=sorted(RAW_EXPOSURE_FEATURES),
        excluded_harmful_features=sorted(HARMFUL_FEATURES),
        lightgbm_params=PARAMS,
        num_boost_round=rounds,
        grouped_cv_folds=n_splits,
        corrected_loco_inner_folds=inner_splits,
        pu_bags=pu_bags,
        unlabeled_per_positive=unlabeled_per_positive,
        grouped_oof_thresholds=thresholds,
        india_policy=(
            "Do not build India common-window features or run India inference "
            "until a foreign-country model and threshold are selected."
        ),
    )
    with open(OUT / "05b_feature_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print("\nCORRECTED LEAVE-ONE-COUNTRY-OUT")
    print(show(loco))
    print("\nTop 20 reduced-model features")
    print(importance.head(20).to_string(index=False))
    return model_results, loco


def main():
    action = sys.argv[1].lower() if len(sys.argv) > 1 else "all"
    if action in {"features", "all"}:
        build_common_features()
    if action in {"evaluate", "all"}:
        run()
    if action not in {"features", "evaluate", "all"}:
        raise SystemExit("usage: kg_05b_robust_tabular.py [features|evaluate|all]")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_05c_balanced_tabular.py
"""Stage 05c: country-balanced training and transferable thresholds.

Stage 05b showed that pooled metrics hide severe country imbalance and that a
coarse threshold grid misses useful operating points in the extreme score tail.
This stage keeps the fixed 2022-2024 features and evaluates four predefined
weighting schemes with exact and country-macro threshold selection.

India is never loaded. The selected foreign-only model and threshold are saved
for a later, separately authorized India run.
"""
import gc
import json
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd

from kg_common import *
from kg_eval import *
from kg_05b_robust_tabular import (
    FEATURE_TAG,
    N_ROUNDS,
    PARAMS,
    WINDOW_YEARS,
    robust_feature_cols,
)


VARIANTS = [
    dict(name="unweighted", country_alpha=0.0, fragment_balance=False),
    dict(name="site_balanced", country_alpha=0.0, fragment_balance=True),
    dict(name="sqrt_country_site", country_alpha=0.5, fragment_balance=True),
    dict(name="equal_country_site", country_alpha=1.0, fragment_balance=True),
]


def sample_weights(df, country_alpha=0.0, fragment_balance=False):
    """Create mean-one training weights without using validation labels.

    ``country_alpha=1`` gives every country equal total source weight. A value
    of 0.5 is the square-root compromise between row weighting and full country
    balancing. Fragment balancing gives each EOG site equal positive mass
    within its country while preserving that country's total positive mass.
    """
    counts = df.country.value_counts()
    n_countries = len(counts)
    base = len(df) / (n_countries * counts)
    country_weight = base.pow(float(country_alpha))
    weight = df.country.map(country_weight).to_numpy(dtype="float64")

    if fragment_balance:
        positive = df.is_eog_flare.eq(1) & df.eog_flare_id.notna()
        pos = df.loc[positive, ["country", "eog_flare_id"]].copy()
        fragment_count = pos.groupby(
            ["country", "eog_flare_id"], observed=True
        ).eog_flare_id.transform("size").to_numpy(dtype="float64")
        country_pos = pos.groupby("country", observed=True).eog_flare_id.transform(
            "size"
        ).to_numpy(dtype="float64")
        country_sites = pos.groupby("country", observed=True).eog_flare_id.transform(
            "nunique"
        ).to_numpy(dtype="float64")
        mean_fragments = country_pos / np.maximum(country_sites, 1.0)
        weight[positive.to_numpy()] *= mean_fragments / fragment_count

    weight /= max(float(weight.mean()), 1e-15)
    return weight.astype("float32")


def _train(X, y, train_idx, weights, rounds=N_ROUNDS, seed=31):
    params = dict(PARAMS)
    params.update(
        seed=seed,
        feature_fraction_seed=seed,
        bagging_seed=seed,
        data_random_seed=seed,
    )
    dataset = lgb.Dataset(
        X[train_idx],
        label=y[train_idx],
        weight=weights[train_idx],
        free_raw_data=True,
    )
    return lgb.train(params, dataset, num_boost_round=rounds)


def cv_predict(df, cols, variant, n_splits=5, rounds=N_ROUNDS, seed=31):
    X = df[cols].to_numpy(dtype="float32")
    y = df.is_eog_flare.to_numpy(dtype="int8")
    oof = np.zeros(len(df), dtype="float64")
    train_s = infer_s = 0.0
    for fold, (train_idx, test_idx) in enumerate(grouped_folds(df, n_splits)):
        t0 = time.time()
        # _train indexes this vector; validation entries are never used.
        weights = np.ones(len(df), dtype="float32")
        weights[train_idx] = sample_weights(
            df.iloc[train_idx],
            country_alpha=variant["country_alpha"],
            fragment_balance=variant["fragment_balance"],
        )
        model = _train(
            X, y, train_idx, weights, rounds=rounds,
            seed=seed + fold * 100,
        )
        train_s += time.time() - t0
        t0 = time.time()
        oof[test_idx] = model.predict(X[test_idx])
        infer_s += time.time() - t0
    return oof, train_s, infer_s


def fit_predict(train_df, test_df, cols, variant, rounds=N_ROUNDS, seed=31):
    X_train = train_df[cols].to_numpy(dtype="float32")
    X_test = test_df[cols].to_numpy(dtype="float32")
    y = train_df.is_eog_flare.to_numpy(dtype="int8")
    weights = sample_weights(
        train_df,
        country_alpha=variant["country_alpha"],
        fragment_balance=variant["fragment_balance"],
    )
    train_idx = np.arange(len(train_df), dtype="int64")
    t0 = time.time()
    model = _train(
        X_train, y, train_idx, weights, rounds=rounds, seed=seed
    )
    train_s = time.time() - t0
    t0 = time.time()
    pred = model.predict(X_test)
    infer_s = time.time() - t0
    return pred, train_s, infer_s


def _f1_arrays_at_thresholds(y, score, thresholds):
    order = np.argsort(score)
    sorted_score = score[order]
    sorted_y = y[order].astype("int64")
    cumulative_pos = np.concatenate([[0], np.cumsum(sorted_y)])
    index = np.searchsorted(sorted_score, thresholds, side="left")
    predicted = len(y) - index
    true_positive = int(sorted_y.sum()) - cumulative_pos[index]
    denominator = predicted + int(sorted_y.sum())
    return np.divide(
        2.0 * true_positive,
        denominator,
        out=np.zeros_like(true_positive, dtype="float64"),
        where=denominator > 0,
    )


def macro_f1_threshold(y, score, countries):
    """Select one threshold that maximizes mean country F1."""
    y = np.asarray(y, dtype="int8")
    score = np.asarray(score, dtype="float64")
    countries = np.asarray(countries)

    candidates = np.unique(score)
    unique_countries = np.unique(countries)
    macro_f1 = np.zeros(len(candidates), dtype="float64")
    for country in unique_countries:
        mask = countries == country
        macro_f1 += (
            _f1_arrays_at_thresholds(y[mask], score[mask], candidates)
            / len(unique_countries)
        )
    index = int(np.nanargmax(macro_f1))
    return float(candidates[index]), float(macro_f1[index])


def country_metrics(df, score, threshold, experiment):
    rows = []
    score = np.asarray(score)
    for country in sorted(df.country.unique()):
        mask = df.country.eq(country).to_numpy()
        y = df.loc[mask, "is_eog_flare"].to_numpy(dtype="int8")
        p = score[mask]
        prevalence = float(y.mean())
        row = metrics(
            y, p, threshold,
            name=experiment,
            extra=dict(
                country=country,
                prevalence=prevalence,
                predicted_positive=int((p >= threshold).sum()),
                prediction_rate=float((p >= threshold).mean()),
            ),
        )
        row["pr_auc_lift"] = (
            row["pr_auc"] / prevalence if prevalence > 0 else np.nan
        )
        rows.append(row)
    return pd.DataFrame(rows)


def active_eog_counts(countries):
    sites = eog_sites(active_years=WINDOW_YEARS)
    return {
        country: int((sites.country == country).sum())
        for country in countries
    }


def site_metrics(df, score, threshold, experiment, totals):
    rows = []
    scored = df[[
        "country", "is_eog_flare", "eog_flare_id"
    ]].copy()
    scored["predicted_positive"] = np.asarray(score) >= threshold
    for country in sorted(scored.country.unique()):
        part = scored[scored.country == country]
        labelled = part[
            part.is_eog_flare.eq(1) & part.eog_flare_id.notna()
        ]
        recoverable = int(labelled.eog_flare_id.nunique())
        detected = int(labelled.loc[
            labelled.predicted_positive, "eog_flare_id"
        ].nunique())
        total = int(totals.get(country, recoverable))
        rows.append(dict(
            experiment=experiment,
            country=country,
            eog_sites_total=total,
            eog_sites_recoverable=recoverable,
            eog_sites_detected=detected,
            recall_of_recoverable=detected / max(recoverable, 1),
            recall_of_all_active=detected / max(total, 1),
        ))
    return pd.DataFrame(rows)


def evaluate_variants(df, cols, n_splits=5, rounds=N_ROUNDS, seed=31):
    y = df.is_eog_flare.to_numpy(dtype="int8")
    countries = df.country.to_numpy()
    totals = active_eog_counts(sorted(df.country.unique()))
    summary_rows = []
    country_tables = []
    site_tables = []
    oof_frame = df[[
        "source_id", "country", "block_id", "lat", "lon",
        "is_eog_flare", "eog_flare_id",
    ]].copy()

    for variant in VARIANTS:
        score, train_s, infer_s = cv_predict(
            df, cols, variant, n_splits=n_splits, rounds=rounds,
            seed=seed,
        )
        pooled_threshold, pooled_f1 = best_f1_threshold(y, score)
        threshold, macro_f1 = macro_f1_threshold(y, score, countries)
        ctable = country_metrics(df, score, threshold, variant["name"])
        stable = site_metrics(
            df, score, threshold, variant["name"], totals
        )
        row = metrics(
            y, score, threshold, train_s, infer_s,
            name=variant["name"],
            extra=dict(
                country_alpha=variant["country_alpha"],
                fragment_balance=variant["fragment_balance"],
                n_features=len(cols),
                threshold_exact=threshold,
                threshold_policy="maximize macro country F1 on grouped OOF",
                macro_country_f1=macro_f1,
                worst_country_f1=float(ctable.f1.min()),
                macro_country_pr_auc=float(ctable.pr_auc.mean()),
                pooled_best_threshold=pooled_threshold,
                pooled_best_f1=pooled_f1,
                macro_site_recall_all=float(
                    stable.recall_of_all_active.mean()
                ),
            ),
        )
        summary_rows.append(row)
        country_tables.append(ctable)
        site_tables.append(stable)
        oof_frame[f"score_{variant['name']}"] = score
        print(
            f"{variant['name']}: macro F1={macro_f1:.4f}, "
            f"pooled F1={row['f1']:.4f}, PR-AUC={row['pr_auc']:.4f}, "
            f"threshold={threshold:.4f}",
            flush=True,
        )

    summary = pd.DataFrame(summary_rows).sort_values(
        ["macro_country_f1", "macro_country_pr_auc"], ascending=False
    ).reset_index(drop=True)
    selected_name = str(summary.iloc[0].experiment)
    selected = next(v for v in VARIANTS if v["name"] == selected_name)
    return (
        summary,
        pd.concat(country_tables, ignore_index=True),
        pd.concat(site_tables, ignore_index=True),
        oof_frame,
        selected,
    )


def corrected_loco(df, cols, variant, inner_splits=3, rounds=N_ROUNDS):
    """Nested LOCO with country-macro threshold selection on inner OOF."""
    rows = []
    country_rows = []
    site_rows = []
    predictions = []
    totals = active_eog_counts(sorted(df.country.unique()))

    for index, country in enumerate(sorted(df.country.unique())):
        train_df = df[df.country != country].reset_index(drop=True)
        test_df = df[df.country == country].reset_index(drop=True)
        inner_score, inner_train_s, _ = cv_predict(
            train_df, cols, variant, n_splits=inner_splits,
            rounds=rounds, seed=5000 + index * 1000,
        )
        inner_y = train_df.is_eog_flare.to_numpy(dtype="int8")
        threshold, inner_macro_f1 = macro_f1_threshold(
            inner_y, inner_score, train_df.country.to_numpy()
        )

        score, final_train_s, infer_s = fit_predict(
            train_df, test_df, cols, variant, rounds=rounds,
            seed=9000 + index * 1000,
        )
        y = test_df.is_eog_flare.to_numpy(dtype="int8")
        row = metrics(
            y, score, threshold,
            train_s=inner_train_s + final_train_s,
            infer_s=infer_s,
            name=f"05c LOCO holdout={country}",
            extra=dict(
                country=country,
                model_variant=variant["name"],
                threshold_policy=(
                    "maximize macro country F1 on grouped inner OOF"
                ),
                inner_macro_country_f1=inner_macro_f1,
                inner_n=len(train_df),
                inner_n_pos=int(inner_y.sum()),
            ),
        )
        rows.append(row)
        country_rows.append(country_metrics(
            test_df, score, threshold, variant["name"]
        ))
        site_rows.append(site_metrics(
            test_df, score, threshold, variant["name"], totals
        ))
        predictions.append(pd.DataFrame({
            "source_id": test_df.source_id,
            "country": country,
            "is_eog_flare": y,
            "eog_flare_id": test_df.eog_flare_id,
            "score": score,
            "threshold": threshold,
            "predicted_positive": score >= threshold,
        }))
        print(
            f"LOCO {country}: threshold={threshold:.4f}, "
            f"PR-AUC={row['pr_auc']:.4f}, F1={row['f1']:.4f}, "
            f"recall={row['recall']:.4f}",
            flush=True,
        )
        del train_df, test_df, inner_score, score
        gc.collect()

    return (
        pd.DataFrame(rows),
        pd.concat(country_rows, ignore_index=True),
        pd.concat(site_rows, ignore_index=True),
        pd.concat(predictions, ignore_index=True),
    )


def run(n_splits=5, inner_splits=3, rounds=N_ROUNDS):
    df = load_features(TRAIN_COUNTRIES, tag=FEATURE_TAG)
    if HOLDOUT in set(df.country):
        raise AssertionError("India entered Stage 05c training data")
    cols = robust_feature_cols(df)

    summary, countries, sites, oof, selected = evaluate_variants(
        df, cols, n_splits=n_splits, rounds=rounds
    )
    summary.to_csv(OUT / "05c_weighting_variants.csv", index=False)
    countries.to_csv(OUT / "05c_country_oof.csv", index=False)
    sites.to_csv(OUT / "05c_site_oof.csv", index=False)
    oof.to_parquet(CACHE / "05c_oof_predictions.parquet", index=False)

    loco, loco_countries, loco_sites, loco_predictions = corrected_loco(
        df, cols, selected, inner_splits=inner_splits, rounds=rounds
    )
    loco.to_csv(OUT / "05c_corrected_loco.csv", index=False)
    loco_countries.to_csv(OUT / "05c_loco_country_metrics.csv", index=False)
    loco_sites.to_csv(OUT / "05c_loco_site_metrics.csv", index=False)
    loco_predictions.to_parquet(
        CACHE / "05c_corrected_loco_predictions.parquet", index=False
    )

    selected_row = summary[summary.experiment == selected["name"]].iloc[0]
    final_threshold = float(selected_row.threshold_exact)
    X = df[cols].to_numpy(dtype="float32")
    y = df.is_eog_flare.to_numpy(dtype="int8")
    weights = sample_weights(
        df,
        country_alpha=selected["country_alpha"],
        fragment_balance=selected["fragment_balance"],
    )
    model = _train(
        X, y, np.arange(len(df), dtype="int64"), weights,
        rounds=rounds, seed=13031,
    )
    model.save_model(str(CACHE / "05c_final_foreign_model.txt"))
    importance = pd.DataFrame({
        "feature": cols,
        "gain": model.feature_importance("gain"),
    }).sort_values("gain", ascending=False)
    importance.to_csv(OUT / "05c_feature_importance.csv", index=False)

    manifest = dict(
        stage="05c_balanced_tabular",
        observation_years=list(WINDOW_YEARS),
        training_countries=TRAIN_COUNTRIES,
        holdout_country=HOLDOUT,
        holdout_loaded=False,
        selected_variant=selected,
        selection_rule=(
            "highest macro country F1 on foreign grouped OOF; "
            "macro country PR-AUC breaks ties"
        ),
        threshold=final_threshold,
        threshold_policy="maximize macro country F1 on foreign grouped OOF",
        features=cols,
        lightgbm_params=PARAMS,
        num_boost_round=rounds,
        grouped_cv_folds=n_splits,
        corrected_loco_inner_folds=inner_splits,
        india_policy=(
            "Do not build or score India until Stage 05c results are reviewed "
            "and this model specification is accepted without retuning."
        ),
    )
    with open(OUT / "05c_model_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print("\nWEIGHTING VARIANTS")
    print(summary[[
        "experiment", "thr", "precision", "recall", "f1", "pr_auc",
        "macro_country_f1", "worst_country_f1", "macro_site_recall_all",
    ]].to_string(index=False))
    print(f"\nSelected variant: {selected['name']}")
    print("\nCORRECTED LOCO")
    print(show(loco))
    return summary, loco


def main():
    if len(sys.argv) > 1:
        raise SystemExit("usage: kg_05c_balanced_tabular.py")
    run()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_05d_nested_tabular.py
"""NB5: fold-local weights and matched-seed, nested country evaluation.

Writes only 05d artifacts. Historical 05b/05c outputs are not overwritten.
India is never loaded. These are development estimates against EOG labels.
"""
import gc
import hashlib
import importlib.metadata
import json
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

import kg_05c_balanced_tabular as balanced
from kg_05b_robust_tabular import FEATURE_TAG, PARAMS, WINDOW_YEARS, robust_feature_cols
from kg_common import CACHE, OUT, TRAIN_COUNTRIES, HOLDOUT
from kg_eval import load_features, metrics

PROTOCOL_VERSION = "05d-nested-v1"


def site_safe_blocks(df):
    """Merge spatial blocks connected by the same known site within a country.

    Unknown sites cannot be protected by this label-based integrity check.
    These group identifiers are never model features.
    """
    keys = pd.MultiIndex.from_frame(df[["country", "block_id"]])
    codes, unique = pd.factorize(keys, sort=True)
    parent = np.arange(len(unique))

    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    positive = df.is_eog_flare.eq(1)
    sites = df.loc[positive, ["country", "eog_flare_id"]].copy()
    sites["block_code"] = codes[positive.to_numpy()]
    for _, part in sites.groupby(["country", "eog_flare_id"], observed=True):
        blocks = part.block_code.unique()
        first = root(blocks[0])
        for block in blocks[1:]:
            parent[root(block)] = first
    roots = np.array([root(i) for i in range(len(unique))])
    mapping = unique.to_frame(index=False)
    mapping["evaluation_block"] = [f"siteblock_{r}" for r in roots]
    revised = df.copy()
    revised["block_id"] = mapping.evaluation_block.to_numpy()[codes]
    return revised, mapping


def validate_features(df, cols):
    required = ["source_id", "country", "block_id", "is_eog_flare"]
    if df[required].isna().any().any() or not df.source_id.is_unique:
        raise ValueError("Source IDs must be unique and required identifiers/labels non-null")
    if set(df.country) != set(TRAIN_COUNTRIES) or HOLDOUT in set(df.country):
        raise ValueError("Expected exactly the six foreign countries; India is forbidden")
    if not df.is_eog_flare.isin([0, 1]).all():
        raise ValueError("Expected binary EOG-match labels")
    if df.loc[df.is_eog_flare.eq(1), "eog_flare_id"].isna().any():
        raise ValueError("Positive source missing its physical EOG site identifier")
    for country, part in df.groupby("country", observed=True):
        if part.is_eog_flare.nunique() != 2:
            raise ValueError(f"Both labels are required for country {country}")
    if not cols or np.isinf(df[cols].to_numpy(dtype="float32")).any():
        raise ValueError("Feature set is empty or contains infinite values")


def nested_loco(df, cols, inner_splits=3, rounds=500, seed=31, checkpoint=None):
    rows, selections, sites, predictions = [], [], [], []
    totals = balanced.active_eog_counts(sorted(df.country.unique()))
    for index, country in enumerate(sorted(df.country.unique())):
        train = df.loc[df.country.ne(country)].reset_index(drop=True)
        test = df.loc[df.country.eq(country)].reset_index(drop=True)
        inner_seed, fit_seed = seed + 10000 + index * 1000, seed + 20000 + index * 1000
        print(f"\nHOLDOUT {country}: selecting from training countries only", flush=True)
        summary, _, _, inner_oof, selected = balanced.evaluate_variants(
            train, cols, n_splits=inner_splits, rounds=rounds, seed=inner_seed,
        )
        winner = summary.loc[summary.experiment.eq(selected["name"])].iloc[0]
        threshold = float(winner.threshold_exact)
        selection = summary.copy()
        selection["held_out_country"] = country
        selection["selected"] = selection.experiment.eq(selected["name"])
        selection["inner_seed"] = inner_seed
        selections.append(selection)
        # Only now predict and evaluate the outer country.
        del inner_oof
        score, train_s, infer_s = balanced.fit_predict(
            train, test, cols, selected, rounds=rounds, seed=fit_seed,
        )
        rows.append(metrics(
            test.is_eog_flare.to_numpy(), score, threshold,
            train_s=float(summary.train_s.sum()) + train_s, infer_s=infer_s,
            name=f"05d holdout={country}", extra={
                "country": country, "model_variant": selected["name"],
                "threshold_exact": threshold, "inner_seed": inner_seed,
                "fit_seed": fit_seed, "inner_macro_f1": float(winner.macro_country_f1),
                "threshold_policy": "training-country grouped OOF macro F1",
            },
        ))
        sites.append(balanced.site_metrics(test, score, threshold, selected["name"], totals))
        prediction = test[["source_id", "country", "block_id", "is_eog_flare", "eog_flare_id"]].copy()
        prediction["score"] = score
        prediction["threshold"] = threshold
        prediction["predicted_positive"] = score >= threshold
        prediction["model_variant"] = selected["name"]
        predictions.append(prediction)
        print(f"{country}: {selected['name']}, F1={rows[-1]['f1']:.4f}, AP={rows[-1]['pr_auc']:.4f}", flush=True)
        if checkpoint is not None:
            checkpoint(pd.DataFrame(rows), pd.concat(selections, ignore_index=True),
                       pd.concat(sites, ignore_index=True), pd.concat(predictions, ignore_index=True))
        del train, test, score
        gc.collect()
    return (pd.DataFrame(rows), pd.concat(selections, ignore_index=True),
            pd.concat(sites, ignore_index=True), pd.concat(predictions, ignore_index=True))


def _save_loco(loco, selections, sites, predictions):
    loco.to_csv(OUT / "05d_nested_loco.csv", index=False)
    selections.to_csv(OUT / "05d_inner_selection.csv", index=False)
    sites.to_csv(OUT / "05d_loco_site_metrics.csv", index=False)
    predictions.to_parquet(CACHE / "05d_loco_predictions.parquet", index=False)


def run(n_splits=5, inner_splits=3, rounds=500, seed=31):
    if n_splits < 2 or inner_splits < 2 or rounds < 1:
        raise ValueError("Need at least two folds and one boosting round")
    manifest_path = OUT / "05d_manifest.json"
    if manifest_path.exists():
        raise FileExistsError("05d run already exists; use a fresh Kaggle session/output directory")
    df = load_features(TRAIN_COUNTRIES, tag=FEATURE_TAG)
    cols = robust_feature_cols(df)
    validate_features(df, cols)
    df, mapping = site_safe_blocks(df)
    mapping.to_csv(OUT / "05d_spatial_group_map.csv", index=False)
    inputs = {}
    for country in TRAIN_COUNTRIES:
        path = CACHE / f"features_{country}_{FEATURE_TAG}.parquet"
        with path.open("rb") as stream:
            inputs[path.name] = hashlib.file_digest(stream, "sha256").hexdigest()
    manifest = {
        "protocol": PROTOCOL_VERSION, "status": "running",
        "holdout_country": HOLDOUT, "holdout_loaded": False,
        "training_countries": TRAIN_COUNTRIES, "years": list(WINDOW_YEARS),
        "features": cols, "n_sources": len(df), "n_positive": int(df.is_eog_flare.sum()),
        "n_splits": n_splits, "inner_splits": inner_splits, "rounds": rounds,
        "seed": seed, "seed_repetitions": 1,
        "seed_policy": "matched across variants; inner=seed+10000+country_index*1000; outer=seed+20000+country_index*1000; CV fold adds fold_index*100",
        "weights": "computed from training fold only",
        "selection": "macro-country F1, then macro-country AP, using only outer training countries",
        "spatial_groups_before": len(mapping),
        "spatial_groups_after": int(mapping.evaluation_block.nunique()),
        "input_sha256": inputs, "python": platform.python_version(),
        "versions": {p: importlib.metadata.version(p) for p in ["numpy", "pandas", "lightgbm", "scikit-learn", "pyarrow"]},
        "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=Path(__file__).parent, text=True).strip(),
        "limitations": "EOG proxy labels; unknown site overlap remains possible; foreign data used in historical research decisions; no India accuracy or repeated-seed uncertainty",
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Loaded {len(df):,} sources, {len(cols)} features; India excluded", flush=True)
    loco, selections, sites, predictions = nested_loco(
        df, cols, inner_splits=inner_splits, rounds=rounds, seed=seed, checkpoint=_save_loco,
    )
    # All-country development selection is separate from the outer evaluation.
    summary, countries, site_oof, oof, selected = balanced.evaluate_variants(
        df, cols, n_splits=n_splits, rounds=rounds, seed=seed,
    )
    summary.to_csv(OUT / "05d_development_variants.csv", index=False)
    countries.to_csv(OUT / "05d_development_countries.csv", index=False)
    site_oof.to_csv(OUT / "05d_development_sites.csv", index=False)
    oof.to_parquet(CACHE / "05d_oof_predictions.parquet", index=False)
    final_seed = seed + 30000
    weights = balanced.sample_weights(df, selected["country_alpha"], selected["fragment_balance"])
    model = balanced._train(df[cols].to_numpy(dtype="float32"), df.is_eog_flare.to_numpy(dtype="int8"),
                            np.arange(len(df)), weights, rounds=rounds, seed=final_seed)
    model.save_model(str(CACHE / "05d_final_foreign_model.txt"))
    pd.DataFrame({"feature": cols, "gain": model.feature_importance("gain")}).sort_values(
        "gain", ascending=False).to_csv(OUT / "05d_feature_importance.csv", index=False)
    final_params = dict(PARAMS)
    final_params.update({key: final_seed for key in ["seed", "feature_fraction_seed", "bagging_seed", "data_random_seed"]})
    manifest.update(status="complete", selected_variant=selected,
                    final_threshold=float(summary.loc[summary.experiment.eq(selected["name"]), "threshold_exact"].iloc[0]),
                    final_lightgbm_params=final_params,
                    macro_loco_f1=float(loco.f1.mean()), macro_loco_pr_auc=float(loco.pr_auc.mean()))
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(loco[["country", "model_variant", "f1", "pr_auc", "recall"]].to_string(index=False))
    return summary, loco


if __name__ == "__main__":
    run()


In [ ]:
%%writefile /kaggle/working/nb8_code/kg_05e_domain_revamp.py
"""Foreign-only domain-generalization revamp for SIH 26162.

India is forbidden. Variants are selected inside each outer country holdout.
Country-relative ranks use feature values only, never labels.
"""
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import platform
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from kg_05b_robust_tabular import FEATURE_TAG, WINDOW_YEARS, robust_feature_cols
from kg_05c_balanced_tabular import (
    active_eog_counts, country_metrics, macro_f1_threshold, site_metrics,
)
from kg_05d_nested_tabular import site_safe_blocks, validate_features
from kg_common import CACHE, HOLDOUT, OUT, TRAIN_COUNTRIES
from kg_eval import metrics


PROTOCOL_VERSION = "05e-domain-revamp-v1"
RANK_BASES = {
    "frp_mean", "frp_max", "frp_med", "frp_p90", "frp_std",
    "frp_dens_mean", "frp_dens_max", "frp_dens_med", "frp_dens_std",
    "t_mir_mean", "t_mir_max", "t_mir_med", "t_mir_std",
    "t_lwir_mean", "t_lwir_max", "t_lwir_med", "t_lwir_std",
    "dt_mir_lwir_mean", "dt_mir_lwir_max", "dt_mir_lwir_med",
    "dt_mir_lwir_std", "frp_sum_per_year",
}
COMPACT_RAW = {
    "active_days_per_year", "active_months_per_year", "det_per_day",
    "det_per_year", "duty_cycle", "span_window_frac", "mean_gap_days",
    "max_gap_days", "modis_per_year", "snpp_per_year", "n_sensors",
    "snpp_modis_ratio", "night_frac", "sat_frac", "lst_mean", "lst_std",
    "frp_cv",
}
PARAMS = {
    "objective": "binary", "learning_rate": 0.05, "num_leaves": 31,
    "max_depth": 8, "min_data_in_leaf": 100, "feature_fraction": 0.8,
    "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l1": 1.0,
    "lambda_l2": 5.0, "max_bin": 127, "verbose": -1,
    "num_threads": -1, "force_col_wise": True,
}
VARIANTS = [
    {"name": "regularized_raw", "schema": "raw", "persistent_train": False},
    {"name": "ranked_physics", "schema": "ranked", "persistent_train": False},
    {"name": "ranked_compact", "schema": "compact", "persistent_train": False},
    {"name": "ranked_compact_persistent", "schema": "compact", "persistent_train": True},
]


def add_country_ranks(df: pd.DataFrame, raw_cols: list[str]) -> tuple[pd.DataFrame, list[str]]:
    if HOLDOUT in set(df.country):
        raise ValueError("India is forbidden during model development")
    ranked = df.copy()
    rank_bases = sorted(set(raw_cols) & RANK_BASES)
    for column in rank_bases:
        ranked[f"{column}_country_pct"] = ranked.groupby(
            "country", observed=True
        )[column].rank(method="average", pct=True).astype("float32")
    return ranked, rank_bases


def feature_schemas(
    df: pd.DataFrame, raw_columns: list[str] | None = None
) -> dict[str, list[str]]:
    raw = list(raw_columns) if raw_columns is not None else robust_feature_cols(df)
    ranked_bases = sorted(set(raw) & RANK_BASES)
    ranked = sorted(
        (set(raw) - set(ranked_bases) - {"pix_km2_mean", "spread_m"})
        | {f"{column}_country_pct" for column in ranked_bases}
    )
    compact = sorted(
        (set(raw) & COMPACT_RAW)
        | {f"{column}_country_pct" for column in ranked_bases}
    )
    schemas = {"raw": raw, "ranked": ranked, "compact": compact}
    for name, columns in schemas.items():
        if not columns or len(columns) != len(set(columns)):
            raise ValueError(f"Invalid {name} schema")
        if {"country", "lat", "lon", "type", "eog_dist_m"} & set(columns):
            raise ValueError(f"Leakage in {name} schema")
    return schemas


def _folds(df: pd.DataFrame, n_splits: int):
    from sklearn.model_selection import GroupKFold
    return GroupKFold(n_splits=n_splits).split(
        df, df.is_eog_flare.to_numpy(), groups=df.block_id.to_numpy()
    )


def _model(df, columns, train_idx, rounds, seed, persistent_train):
    train_idx = np.asarray(train_idx, dtype="int64")
    if persistent_train:
        train_idx = train_idx[df.iloc[train_idx].n_days.to_numpy() >= 2]
    y = df.is_eog_flare.to_numpy(dtype="int8")
    if y[train_idx].sum() == 0:
        raise ValueError("Training fold has no positives")
    params = dict(PARAMS)
    params.update({key: seed for key in [
        "seed", "feature_fraction_seed", "bagging_seed", "data_random_seed"
    ]})
    dataset = lgb.Dataset(
        df.iloc[train_idx][columns].to_numpy(dtype="float32"),
        label=y[train_idx], free_raw_data=True,
    )
    return lgb.train(params, dataset, num_boost_round=rounds), len(train_idx)


def cv_predict(df, columns, variant, n_splits=3, rounds=400, seed=71):
    score = np.empty(len(df), dtype="float64")
    train_rows = []
    for fold, (train_idx, test_idx) in enumerate(_folds(df, n_splits)):
        model, used = _model(
            df, columns, train_idx, rounds, seed + fold * 100,
            variant["persistent_train"],
        )
        score[test_idx] = model.predict(
            df.iloc[test_idx][columns].to_numpy(dtype="float32")
        )
        train_rows.append(used)
    return score, train_rows


def ensemble_predict(train, test, columns, variant, rounds, seeds):
    score = np.zeros(len(test), dtype="float64")
    used = []
    for seed in seeds:
        model, count = _model(
            train, columns, np.arange(len(train)), rounds, seed,
            variant["persistent_train"],
        )
        score += model.predict(test[columns].to_numpy(dtype="float32")) / len(seeds)
        used.append(count)
    return score, used


def evaluate_variants(df, schemas, n_splits=3, rounds=400, seed=71):
    rows, predictions = [], {}
    y = df.is_eog_flare.to_numpy(dtype="int8")
    countries = df.country.to_numpy()
    for variant in VARIANTS:
        columns = schemas[variant["schema"]]
        started = time.time()
        score, train_rows = cv_predict(
            df, columns, variant, n_splits=n_splits, rounds=rounds, seed=seed
        )
        threshold, macro_f1 = macro_f1_threshold(y, score, countries)
        country = country_metrics(df, score, threshold, variant["name"])
        row = metrics(y, score, threshold, name=variant["name"], extra={
            "schema": variant["schema"],
            "persistent_train": variant["persistent_train"],
            "n_features": len(columns), "threshold_exact": threshold,
            "macro_country_f1": macro_f1,
            "macro_country_pr_auc": float(country.pr_auc.mean()),
            "worst_country_pr_auc": float(country.pr_auc.min()),
            "mean_training_rows": float(np.mean(train_rows)),
            "elapsed_s": time.time() - started,
        })
        rows.append(row)
        predictions[variant["name"]] = score
        print(
            f"{variant['name']}: macro AP={row['macro_country_pr_auc']:.4f}, "
            f"macro F1={macro_f1:.4f}, pooled AP={row['pr_auc']:.4f}", flush=True
        )
    summary = pd.DataFrame(rows).sort_values(
        ["macro_country_pr_auc", "macro_country_f1"], ascending=False
    ).reset_index(drop=True)
    winner = next(v for v in VARIANTS if v["name"] == summary.iloc[0].experiment)
    return summary, winner, predictions


def nested_loco(df, schemas, inner_splits=3, rounds=400, seed=71, checkpoint=None):
    results, selections, predictions = [], [], []
    totals = active_eog_counts(sorted(df.country.unique()))
    site_rows = []
    for index, country in enumerate(sorted(df.country.unique())):
        train = df[df.country.ne(country)].reset_index(drop=True)
        test = df[df.country.eq(country)].reset_index(drop=True)
        inner_seed = seed + 10_000 + index * 1_000
        summary, winner, _ = evaluate_variants(
            train, schemas, inner_splits, rounds, inner_seed
        )
        selected = summary.iloc[0]
        summary["held_out_country"] = country
        summary["selected"] = summary.experiment.eq(winner["name"])
        selections.append(summary)
        seeds = [seed + 20_000 + index * 1_000 + offset for offset in (0, 101, 202)]
        columns = schemas[winner["schema"]]
        score, used = ensemble_predict(train, test, columns, winner, rounds, seeds)
        threshold = float(selected.threshold_exact)
        results.append(metrics(
            test.is_eog_flare.to_numpy(dtype="int8"), score, threshold,
            name=f"05e holdout={country}", extra={
                "country": country, "model_variant": winner["name"],
                "threshold_exact": threshold, "n_features": len(columns),
                "ensemble_seeds": json.dumps(seeds),
                "training_rows_per_seed": json.dumps(used),
                "inner_macro_pr_auc": float(selected.macro_country_pr_auc),
                "selection_policy": "training-country macro AP then macro F1",
            }
        ))
        site_rows.append(site_metrics(test, score, threshold, winner["name"], totals))
        part = test[["source_id", "country", "block_id", "is_eog_flare", "eog_flare_id"]].copy()
        part["score"] = score
        part["threshold"] = threshold
        part["predicted_positive"] = score >= threshold
        part["model_variant"] = winner["name"]
        predictions.append(part)
        print(f"HOLDOUT {country}: {winner['name']}, F1={results[-1]['f1']:.4f}, AP={results[-1]['pr_auc']:.4f}", flush=True)
        if checkpoint:
            checkpoint(pd.DataFrame(results), pd.concat(selections), pd.concat(site_rows), pd.concat(predictions))
        del train, test, score
        gc.collect()
    return pd.DataFrame(results), pd.concat(selections), pd.concat(site_rows), pd.concat(predictions)


def run(input_dir="/kaggle/input", root="/kaggle/working/nb8_domain_revamp", inner_splits=3, rounds=400, seed=71):
    root = Path(root)
    output = root / "outputs"
    cache = root / "cache"
    output.mkdir(parents=True, exist_ok=True)
    cache.mkdir(parents=True, exist_ok=True)
    manifest_path = output / "05e_manifest.json"
    if manifest_path.exists():
        raise FileExistsError("Use a fresh output directory for a full run")
    input_dir = Path(input_dir)
    frames, hashes = [], {}
    for country in TRAIN_COUNTRIES:
        name = f"features_{country}_{FEATURE_TAG}.parquet"
        matches = list(input_dir.rglob(name))
        if len(matches) != 1:
            raise FileNotFoundError(f"Need exactly one {name}; found {matches}")
        with matches[0].open("rb") as stream:
            hashes[name] = hashlib.file_digest(stream, "sha256").hexdigest()
        frames.append(pd.read_parquet(matches[0]))
    df = pd.concat(frames, ignore_index=True)
    raw = robust_feature_cols(df)
    validate_features(df, raw)
    df, mapping = site_safe_blocks(df)
    df, rank_bases = add_country_ranks(df, raw)
    schemas = feature_schemas(df, raw)
    mapping.to_csv(output / "05e_spatial_group_map.csv", index=False)
    manifest = {
        "protocol": PROTOCOL_VERSION, "status": "running", "holdout_country": HOLDOUT,
        "holdout_loaded": False, "countries": TRAIN_COUNTRIES, "years": list(WINDOW_YEARS),
        "n_sources": len(df), "n_positive": int(df.is_eog_flare.sum()),
        "rank_features": rank_bases, "schemas": schemas, "variants": VARIANTS,
        "params": PARAMS, "inner_splits": inner_splits, "rounds": rounds, "seed": seed,
        "outer_ensemble_seeds": 3, "selection": "nested macro-country AP then macro-country F1",
        "rank_policy": "label-free within-country empirical percentile, batch-country inference",
        "input_sha256": hashes, "python": platform.python_version(),
        "versions": {p: importlib.metadata.version(p) for p in ["numpy", "pandas", "lightgbm", "scikit-learn", "pyarrow"]},
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    def save(loco, selections, sites, preds):
        loco.to_csv(output / "05e_nested_loco.csv", index=False)
        selections.to_csv(output / "05e_inner_selection.csv", index=False)
        sites.to_csv(output / "05e_loco_site_metrics.csv", index=False)
        preds.to_parquet(cache / "05e_loco_predictions.parquet", index=False)
    loco, selections, sites, preds = nested_loco(df, schemas, inner_splits, rounds, seed, save)
    development, winner, oof = evaluate_variants(df, schemas, 5, rounds, seed)
    development.to_csv(output / "05e_development_variants.csv", index=False)
    oof_frame = df[["source_id", "country", "block_id", "is_eog_flare", "eog_flare_id"]].copy()
    for name, score in oof.items():
        oof_frame[f"score_{name}"] = score
    oof_frame.to_parquet(cache / "05e_oof_predictions.parquet", index=False)
    columns = schemas[winner["schema"]]
    gains = np.zeros(len(columns))
    final_seeds = [seed + 30_000 + offset for offset in (0, 101, 202)]
    for model_index, final_seed in enumerate(final_seeds):
        model, _ = _model(df, columns, np.arange(len(df)), rounds, final_seed, winner["persistent_train"])
        model.save_model(str(cache / f"05e_final_model_{model_index}.txt"))
        gains += model.feature_importance("gain") / len(final_seeds)
    pd.DataFrame({"feature": columns, "mean_gain": gains}).sort_values("mean_gain", ascending=False).to_csv(output / "05e_feature_importance.csv", index=False)
    manifest.update({
        "status": "complete", "selected_variant": winner,
        "selected_features": columns, "final_seeds": final_seeds,
        "final_threshold": float(development.iloc[0].threshold_exact),
        "macro_loco_f1": float(loco.f1.mean()), "macro_loco_pr_auc": float(loco.pr_auc.mean()),
    })
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return development, loco


if __name__ == "__main__":
    run()


## Attach inputs

Attach the full `sih nb2` saved output and the original SIH dataset containing the EOG workbook. Do not attach or load India-specific evaluation outputs.


In [ ]:
import os

INPUT = Path('/kaggle/input')
workbook_name = 'Flare-Volume-Estimates-by-individual-Flare-Location-2012-2025.xlsx'
workbooks = list(INPUT.rglob(workbook_name))
assert len(workbooks) == 1, f'Need exactly one original EOG workbook; found {workbooks}'
os.environ['SIH_DATA'] = str(workbooks[0].parents[2])

for country in ['Iraq', 'Algeria', 'Nigeria', 'Libya', 'Angola', 'Indonesia']:
    name = f'features_{country}_2022_2024.parquet'
    matches = list(INPUT.rglob(name))
    assert len(matches) == 1, f'Need exactly one {name}; found {matches}'
    print(country, matches[0])

assert not list(INPUT.rglob('features_India_2022_2024.parquet')), 'Remove any India common-window input from this development run'
sys.path.insert(0, str(CODE))
from kg_05e_domain_revamp import PROTOCOL_VERSION, run
assert PROTOCOL_VERSION == '05e-domain-revamp-v1'
print('Ready:', PROTOCOL_VERSION)


## Run the complete nested experiment

This is CPU work. Partial outer-country results are checkpointed, but automatic resume is not implemented. Do not select a model from a partial table.


In [ ]:
development_results, country_holdout_results = run(
    input_dir=INPUT,
    root='/kaggle/working/nb8_domain_revamp',
    inner_splits=3,
    rounds=400,
    seed=71,
)

display(development_results)
display(country_holdout_results)


In [ ]:
import json
import pandas as pd

ROOT = Path('/kaggle/working/nb8_domain_revamp')
manifest = json.loads((ROOT / 'outputs/05e_manifest.json').read_text())
assert manifest['status'] == 'complete'
assert manifest['holdout_loaded'] is False
assert manifest['holdout_country'] == 'India'

loco = pd.read_csv(ROOT / 'outputs/05e_nested_loco.csv')
assert len(loco) == 6
display(loco[['country', 'model_variant', 'precision', 'recall', 'f1', 'pr_auc', 'roc_auc']])
print('Macro held-out-country F1:', loco.f1.mean())
print('Macro held-out-country AP:', loco.pr_auc.mean())
print('Selected final variant:', manifest['selected_variant'])
print('Selected features:', manifest['selected_features'])


## Package all necessary results


In [ ]:
import zipfile
from IPython.display import FileLink

ROOT = Path('/kaggle/working/nb8_domain_revamp')
required_outputs = [
    '05e_manifest.json', '05e_spatial_group_map.csv', '05e_nested_loco.csv',
    '05e_inner_selection.csv', '05e_loco_site_metrics.csv',
    '05e_development_variants.csv', '05e_feature_importance.csv',
]
required_cache = [
    '05e_loco_predictions.parquet', '05e_oof_predictions.parquet',
    '05e_final_model_0.txt', '05e_final_model_1.txt', '05e_final_model_2.txt',
]
files = [(ROOT / 'outputs' / name, f'outputs/{name}') for name in required_outputs]
files += [(ROOT / 'cache' / name, f'cache/{name}') for name in required_cache]
files += [(path, f'code/{path.name}') for path in sorted(CODE.glob('*.py'))]
for file, _ in files:
    assert file.is_file(), f'Missing required artifact: {file}'

bundle = Path('/kaggle/working/nb8_domain_revamp.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file, archive_name in files:
        archive.write(file, arcname=archive_name)
print(bundle, f'{bundle.stat().st_size / 1024**2:.1f} MiB')
display(FileLink(str(bundle)))
